In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import random
from numba import jit

In [ ]:
@jit(nopython=True)
def dla_simulation(size, n_particles, prob, m_thresh, interval):
    num_frames = n_particles // interval + 1
    history = np.zeros((num_frames, size, size), dtype=np.int8)

    grid = np.zeros((size, size), dtype=np.int8)
    visit_counts = np.zeros((size, size), dtype=np.int8)
    adjacency_map = np.zeros((size, size), dtype=np.int8)
    
    center = size // 2
    grid[center, center] = 1
    history[0] = grid.copy()
    frame_idx = 1

    # Inicjalizacja sąsiadów zarodka
    moves = np.array([[0, 1], [0, -1], [1, 0], [-1, 0]])
    for i in range(4):
        adjacency_map[center + moves[i, 0], center + moves[i, 1]] = 1

    # Aktualny promień agregatu
    rmax = 0
    particles_added = 0

    while particles_added < n_particles:
        start_radius = rmax + 15
        kill_radius = rmax + 100

        if kill_radius > size // 2 - 2:
            kill_radius = size // 2 - 2

        # Koordynaty tworzonej cząstki
        angle = 2 * np.pi * random.random()
        px = int(center + start_radius * np.cos(angle))
        py = int(center + start_radius * np.sin(angle))
        
        while True:
            # Sprawdzenie czy cząstka nie wyszła za daleko
            if (px - center)**2 + (py - center)**2 > kill_radius**2:
                break
            
            # Sprawdzenie czy cząstka dotyka agregatu
            if adjacency_map[px, py] == 1:

                if random.random() <= prob:
                    visit_counts[px, py] += 1
                    
                    if visit_counts[px, py] >= m_thresh:
                        # --- PRZYŁĄCZENIE DO AGREGATU ---
                        grid[px, py] = 1
                        particles_added += 1
                        adjacency_map[px, py] = 0
                        
                        # Aktualizacja promienia
                        dist = np.sqrt((px - center)**2 + (py - center)**2)
                        if dist > rmax:
                            rmax = dist
                        
                        # Aktualizacja mapy sąsiedztwa
                        for i in range(4):
                            nx, ny = px + moves[i, 0], py + moves[i, 1]
                            if 0 <= nx < size and 0 <= ny < size:
                                if grid[nx, ny] == 0:
                                    adjacency_map[nx, ny] = 1

                        # Zapisanie klatki
                        if particles_added % interval == 0:
                            history[frame_idx] = grid.copy()
                            frame_idx += 1

                        # if particles_added % 1000 == 0:
                        #     print(f"{particles_added}")
                        
                        # Cząstka przyłączona, agregat się powiększył
                        break
                    
                    else:
                        # visit_counts < M, cząstka ginie
                        break 
                
                else:
                    # Odbicie
                    pass

            move_idx = random.randint(0, 3)
            nx, ny = px + moves[move_idx, 0], py + moves[move_idx, 1]
            
            if 0 <= nx < size and 0 <= ny < size:
                if grid[nx, ny] == 0:
                    px, py = nx, ny

    return history

In [ ]:
L = 1000
N_PARTICLES = 40000
SNAPSHOT_INTERVAL = 50

PROBABILITY = 0.4
M_THRESHOLD = 3

history_stack = dla_simulation(L, N_PARTICLES, PROBABILITY, M_THRESHOLD, SNAPSHOT_INTERVAL)

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')
title = ax.set_title(f"DLA Growth (Particles: 0)")

img = ax.imshow(history_stack[0], cmap='binary', interpolation='none', vmin=0, vmax=1)

def update(frame_idx):
    current_grid = history_stack[frame_idx]
    img.set_data(current_grid)
    current_particles = frame_idx * SNAPSHOT_INTERVAL
    title.set_text(f"DLA Growth for p = {PROBABILITY}, M = {M_THRESHOLD} (Particles: {current_particles})")
    return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(history_stack),
    interval=50,
    blit=True
)

plt.show()

save_path = f"../results/gifs/10_aggregation_growth_p={str(PROBABILITY).replace('.','')}_M={M_THRESHOLD}.gif"
anim.save(save_path, writer='pillow', fps=20)